In [3]:
import pandas as pd

# Assuming 'data' is already loaded and initial preprocessing is done
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

# Load the dataset
data = pd.read_csv("C:\\Users\\Admin\Downloads\\Side Projects\\Golf Predicitve Model Iteration 2\\Final Package for Sharing\\ASA All PGA Raw Data - Tourn Level.csv")


#count the amount of missing data points in sg_total column
missing_values_count = data['sg_total'].isnull().sum()
print(missing_values_count)



7683


In [4]:
# Fill missing sg_total values with the average sg_total for the same position
data['sg_total'] = data.groupby('pos')['sg_total'].transform(lambda x: x.fillna(x.mean()))

# Fill any remaining NaNs in sg_total with the overall mean of sg_total
if data['sg_total'].isnull().any():
    data['sg_total'].fillna(data['sg_total'].mean(), inplace=True)

In [5]:

import statsmodels.formula.api as smf
# Define the fixed effects model using Q() for quoting variable names
fe_formula_sg = 'sg_total ~ C(Q("player id")) + C(Q("tournament id"))'

# Fit the model
fe_model_sg = smf.ols(fe_formula_sg, data=data).fit()

# Calculate adjusted sg_total
data['adjusted_sg_total'] = data['sg_total'] - fe_model_sg.fittedvalues


In [6]:
# Ensure the date column is in datetime format (reconfirming)
data['date'] = pd.to_datetime(data['date'])


# Print data structure to confirm the status of 'player id' and 'date'
print("Data columns before grouping:", data.columns)
print("Index before grouping:", data.index)

# Sort data by player and date to ensure correct rolling calculations
data.sort_values(by=['player id', 'date'], inplace=True)



Data columns before grouping: Index(['Player_initial_last', 'tournament id', 'player id', 'hole_par',
       'strokes', 'hole_DKP', 'hole_FDP', 'hole_SDP', 'streak_DKP',
       'streak_FDP', 'streak_SDP', 'n_rounds', 'made_cut', 'pos', 'finish_DKP',
       'finish_FDP', 'finish_SDP', 'total_DKP', 'total_FDP', 'total_SDP',
       'player', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'tournament name',
       'course', 'date', 'purse', 'season', 'no_cut', 'Finish', 'sg_putt',
       'sg_arg', 'sg_app', 'sg_ott', 'sg_t2g', 'sg_total',
       'adjusted_sg_total'],
      dtype='object')
Index before grouping: RangeIndex(start=0, stop=36864, step=1)


In [7]:
# Define a function to calculate exponentially weighted averages
def calculate_ewm_averages(group):
    group.set_index('date', inplace=True)
    
    # Calculate the 2-Year Exponentially Weighted Moving Average
    group['2yr_ewm_sg'] = group['adjusted_sg_total'].ewm(span=730, adjust=False).mean()
    
    # Calculate the 1-Year Exponentially Weighted Moving Average
    group['1yr_ewm_sg'] = group['adjusted_sg_total'].ewm(span=365, adjust=False).mean()
    
    # Calculate the 2-Month Exponentially Weighted Moving Average
    group['2mo_ewm_sg'] = group['adjusted_sg_total'].ewm(span=60, adjust=False).mean()
    
    group.reset_index(inplace=True)  # Reset index to bring 'date' back as a column
    return group

# Apply the function to each group, ensuring 'player id' is treated as a column
data = data.groupby('player id').apply(calculate_ewm_averages).reset_index(drop=True)


In [8]:
# Define a function to calculate exponentially weighted averages
def calculate_ewm_averages(group):
    group.set_index('date', inplace=True)
    
    # Calculate the 2-Year Exponentially Weighted Moving Average
    group['2yr_ewm_sg'] = group['adjusted_sg_total'].ewm(span=730, adjust=False).mean()
    
    # Calculate the 1-Year Exponentially Weighted Moving Average
    group['1yr_ewm_sg'] = group['adjusted_sg_total'].ewm(span=365, adjust=False).mean()
    
    # Calculate the 2-Month Exponentially Weighted Moving Average
    group['2mo_ewm_sg'] = group['adjusted_sg_total'].ewm(span=60, adjust=False).mean()
    
    group.reset_index(inplace=True)  # Reset index to bring 'date' back as a column
    return group

# Apply the function to each group, ensuring 'player id' is treated as a column
data = data.groupby('player id').apply(calculate_ewm_averages).reset_index(drop=True)


In [9]:
# Print structure after processing to check consistency
print("Data columns after grouping and processing:", data.columns)
print("Index after grouping and processing:", data.index)

Data columns after grouping and processing: Index(['date', 'Player_initial_last', 'tournament id', 'player id', 'hole_par',
       'strokes', 'hole_DKP', 'hole_FDP', 'hole_SDP', 'streak_DKP',
       'streak_FDP', 'streak_SDP', 'n_rounds', 'made_cut', 'pos', 'finish_DKP',
       'finish_FDP', 'finish_SDP', 'total_DKP', 'total_FDP', 'total_SDP',
       'player', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'tournament name',
       'course', 'purse', 'season', 'no_cut', 'Finish', 'sg_putt', 'sg_arg',
       'sg_app', 'sg_ott', 'sg_t2g', 'sg_total', 'adjusted_sg_total',
       '2yr_ewm_sg', '1yr_ewm_sg', '2mo_ewm_sg'],
      dtype='object')
Index after grouping and processing: RangeIndex(start=0, stop=36864, step=1)


In [10]:
# Calculate the last event average
try:
    data['last_event_avg_sg'] = data.groupby('player id')['adjusted_sg_total'].shift(1)
    print("Last event averages calculated successfully.")
except Exception as e:
    print("Error calculating last event averages:", e)

Last event averages calculated successfully.


In [11]:
# Add rolling statistics
data['rolling_mean_3'] = data.groupby('player id')['adjusted_sg_total'].transform(lambda x: x.rolling(3, min_periods=1).mean())
data['rolling_std_3'] = data.groupby('player id')['adjusted_sg_total'].transform(lambda x: x.rolling(3, min_periods=1).std())
data['rolling_mean_6'] = data.groupby('player id')['adjusted_sg_total'].transform(lambda x: x.rolling(6, min_periods=1).mean())
data['rolling_std_6'] = data.groupby('player id')['adjusted_sg_total'].transform(lambda x: x.rolling(6, min_periods=1).std())

# Print structure after processing to check consistency
print("Data columns after grouping and processing:", data.columns)
print("Index after grouping and processing:", data.index)

# Final check of data structure
print(data.head())

# Select only the required features
features = ['2yr_ewm_sg', '2mo_ewm_sg', 'last_event_avg_sg',
            'rolling_mean_3', 'rolling_std_3',
            'rolling_mean_6', 'rolling_std_6']

Data columns after grouping and processing: Index(['date', 'Player_initial_last', 'tournament id', 'player id', 'hole_par',
       'strokes', 'hole_DKP', 'hole_FDP', 'hole_SDP', 'streak_DKP',
       'streak_FDP', 'streak_SDP', 'n_rounds', 'made_cut', 'pos', 'finish_DKP',
       'finish_FDP', 'finish_SDP', 'total_DKP', 'total_FDP', 'total_SDP',
       'player', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'tournament name',
       'course', 'purse', 'season', 'no_cut', 'Finish', 'sg_putt', 'sg_arg',
       'sg_app', 'sg_ott', 'sg_t2g', 'sg_total', 'adjusted_sg_total',
       '2yr_ewm_sg', '1yr_ewm_sg', '2mo_ewm_sg', 'last_event_avg_sg',
       'rolling_mean_3', 'rolling_std_3', 'rolling_mean_6', 'rolling_std_6'],
      dtype='object')
Index after grouping and processing: RangeIndex(start=0, stop=36864, step=1)
        date Player_initial_last  tournament id  player id  hole_par  strokes  \
0 2014-10-12          R. Allenby           2271          5       288      277   
1 2014-10-19        

In [10]:
import pandas as pd
import numpy as np
from sklearn.metrics import brier_score_loss, log_loss
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

# Assume missing position values indicate the player missed the cut
data['pos'].fillna(value=999, inplace=True)  # Use a value that guarantees classification as 'Missed Cut'

# Define binary targets for each category
data['Top 10'] = (data['pos'] <= 10).astype(int)
data['Top 20'] = (data['pos'] <= 20).astype(int)
data['Top 40'] = (data['pos'] <= 40).astype(int)
data['Made Cut'] = (data['pos'] <= 70).astype(int)
data['Missed Cut'] = (data['pos'] > 70).astype(int)

# Define the hyperparameter grid for Logistic Regression
param_grid = {
    'C': [0.01, 0.1, 1, 10, 100],  # Regularization strength
    'penalty': ['l2'],             # Only l2 penalty is supported by default for LogisticRegression solver 'lbfgs'
    'solver': ['lbfgs'],
    'max_iter': [1000]             # Increase max iterations for convergence
}

# Placeholders for overall metrics
overall_brier_scores = {category: [] for category in ['Top 10', 'Top 20', 'Top 40', 'Made Cut', 'Missed Cut']}
overall_log_losses = {category: [] for category in ['Top 10', 'Top 20', 'Top 40', 'Made Cut', 'Missed Cut']}

# Initialize an empty DataFrame to store results across all players
all_test_results = pd.DataFrame()

# Loop through each player in the dataset
for player_id in data['player id'].unique():
    player_data = data[data['player id'] == player_id]
    
    # Remove rows with any missing data in the relevant columns
    player_data = player_data.dropna(subset=['2yr_ewm_sg', '2mo_ewm_sg','1yr_ewm_sg',
                                             'rolling_mean_3', 'rolling_std_3',
                                             'rolling_mean_6', 'rolling_std_6'])
    
    # Determine the split point for 70% training and 30% testing
    split_point = int(len(player_data) * 0.7)
    
    # Ensure there is enough data for training and testing
    if len(player_data) > 1 and split_point < len(player_data):
        # Split the data: first 70% rounds for training, last 30% rounds for testing
        train_data = player_data.iloc[:split_point]  # First 70% rounds for training
        test_data = player_data.iloc[split_point:]   # Last 30% rounds for testing
        
        # Prepare features and target for the player's model
        X_train = train_data[['2yr_ewm_sg', '2mo_ewm_sg','1yr_ewm_sg', 
                              'rolling_mean_3', 'rolling_std_3',
                              'rolling_mean_6', 'rolling_std_6']]
        X_test = test_data[['2yr_ewm_sg', '2mo_ewm_sg','1yr_ewm_sg',
                            'rolling_mean_3', 'rolling_std_3',
                            'rolling_mean_6', 'rolling_std_6']]
        
        # Initialize a DataFrame to store test results for this player
        test_results = test_data[['player id']].copy()
        
        # Train and evaluate for each category using Logistic Regression with hyperparameter tuning
        for category in ['Top 10', 'Top 20', 'Top 40', 'Made Cut', 'Missed Cut']:
            y_train = train_data[category]
            y_test = test_data[category]
            
            # Check if both classes are present in the training data
            if len(np.unique(y_train)) > 1:  # Proceed only if there are both 0s and 1s
                # Get class distribution and check if we can use 5-fold cross-validation
                class_counts = np.bincount(y_train)
                min_class_size = np.min(class_counts)
                
                if min_class_size >= 5:
                    # Use 5-fold cross-validation
                    cv_folds = 5
                else:
                    # If we don't have enough samples for 5-fold, use `min_class_size` folds
                    cv_folds = min_class_size
                
                if cv_folds >= 2:
                    # Perform hyperparameter tuning with GridSearchCV using cross-validation
                    model = GridSearchCV(LogisticRegression(), param_grid, scoring='neg_log_loss', cv=cv_folds)
                    model.fit(X_train, y_train)
                else:
                    # If there are fewer than 2 samples in any class, skip cross-validation
                    model = LogisticRegression(max_iter=1000)
                    model.fit(X_train, y_train)
                
                # Predict the probabilities for the last 30% of rounds
                probabilities = model.predict_proba(X_test)[:, 1]  # Probabilities for the positive class
                
                # Add the predictions to the test_results DataFrame
                test_results[f'{category}_probability'] = probabilities
                test_results[f'{category}_actual'] = y_test.values
                
                # Calculate Brier Score for this player and category
                brier_score = brier_score_loss(y_test, probabilities)
                overall_brier_scores[category].append(brier_score)
                
                # Calculate Log Loss for this player and category, with a check for single-class test sets
                if len(np.unique(y_test)) > 1:  # Check if there are both 0s and 1s in y_test
                    log_loss_value = log_loss(y_test, probabilities)
                else:
                    log_loss_value = np.nan  # Assign NaN if only one class is present
                overall_log_losses[category].append(log_loss_value)
        
        # Append the test results for this player to the overall results
        all_test_results = pd.concat([all_test_results, test_results], ignore_index=True)

# Export the final results to a CSV file
all_test_results.to_csv('player_test_results.csv', index=False)

# Display overall average results
for category in ['Top 10', 'Top 20', 'Top 40', 'Made Cut', 'Missed Cut']:
    avg_brier_score = np.nanmean(overall_brier_scores[category])  # Use np.nanmean to ignore NaN values
    avg_log_loss = np.nanmean(overall_log_losses[category])  # Use np.nanmean to ignore NaN values
    print(f"Category: {category}")
    print(f"  Average Brier Score: {avg_brier_score:.4f}")
    print(f"  Average Log Loss: {avg_log_loss:.4f}")
    print('-' * 30)


Category: Top 10
  Average Brier Score: 0.0584
  Average Log Loss: 0.2842
------------------------------
Category: Top 20
  Average Brier Score: 0.1015
  Average Log Loss: 0.4020
------------------------------
Category: Top 40
  Average Brier Score: 0.1745
  Average Log Loss: 0.5594
------------------------------
Category: Made Cut
  Average Brier Score: 0.2422
  Average Log Loss: 0.7029
------------------------------
Category: Missed Cut
  Average Brier Score: 0.2422
  Average Log Loss: 0.7029
------------------------------
